# 🌿 Plant Disease Detection – Exploratory Notebook

This notebook walks through the full pipeline interactively:
dataset exploration → preprocessing → model training → evaluation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from src.utils import set_seed, setup_logging, save_class_names, load_class_names
from src.preprocess import load_dataset, build_dataframe, split_dataset, create_data_generators, visualize_samples
from src.train import build_cnn_model, compile_model
from src.evaluate import plot_training_curves, plot_confusion_matrix, plot_roc_curves

setup_logging()
set_seed(42)
print('TensorFlow:', tf.__version__)

## 1  Dataset Overview

In [ ]:
DATASET_DIR = '../dataset/PlantVillage'

image_paths, labels, class_names = load_dataset(DATASET_DIR)
df = build_dataframe(image_paths, labels)

print(f'Total images : {len(df)}')
print(f'Classes      : {len(class_names)}')
df['label'].value_counts().head(10)

In [ ]:
# Class distribution bar chart
counts = df['label'].value_counts()
plt.figure(figsize=(16, 5))
counts.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Class Distribution', fontsize=14)
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=60, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
visualize_samples(image_paths, labels, n=12)

## 2  Preprocessing & Generators

In [ ]:
splits = split_dataset(image_paths, labels)

train_df = pd.DataFrame({'image_path': splits['train_paths'], 'label': splits['train_labels']})
val_df   = pd.DataFrame({'image_path': splits['val_paths'],   'label': splits['val_labels']})
test_df  = pd.DataFrame({'image_path': splits['test_paths'],  'label': splits['test_labels']})

train_gen, val_gen, test_gen = create_data_generators(train_df, val_df, test_df)
print('Steps per epoch:', len(train_gen))

In [ ]:
# Preview one augmented batch
batch_images, batch_labels = next(train_gen)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, img in zip(axes.flatten(), batch_images[:8]):
    ax.imshow(img)
    ax.axis('off')
plt.suptitle('Augmented Training Batch', fontsize=13)
plt.tight_layout()
plt.show()

## 3  Model Architecture

In [ ]:
model = build_cnn_model(num_classes=len(class_names))
model = compile_model(model)
model.summary()

## 4  Training (short demo – 3 epochs)

In [ ]:
# For a full run use main.py; this cell is just a quick sanity check
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=3,
    verbose=1,
)

In [ ]:
plot_training_curves(history.history)

## 5  Evaluation (load best saved model)

In [ ]:
from src.utils import load_model
from src.evaluate import get_predictions, compute_metrics, plot_confusion_matrix

saved_model = load_model('../saved_model/best_model.keras')
class_names = load_class_names('../saved_model/class_names.json')

y_true, y_pred, y_proba = get_predictions(saved_model, test_gen)
results = compute_metrics(y_true, y_pred, class_names)

In [ ]:
plot_confusion_matrix(y_true, y_pred, class_names)

In [ ]:
plot_roc_curves(y_true, y_proba, class_names)

## 6  Single-image prediction

In [ ]:
from src.predict import predict_single, display_prediction

# Replace with any leaf image path
TEST_IMAGE = splits['test_paths'][0]
result = predict_single(TEST_IMAGE, saved_model, class_names)
display_prediction(TEST_IMAGE, result)